# 01. Carga, limpieza y normalización de datos

Este notebook corresponde a la primera fase práctica del TFM. Su objetivo es cargar los datasets históricos de boyas de Puertos del Estado, revisar su estructura, limpiar valores nulos, normalizar variables y construir los datasets de trabajo que se utilizarán en los análisis posteriores.

El trabajo se organiza en dos datasets procesados:

- `dataset_principal_oleaje.csv`: incluye las variables principales de oleaje de las cuatro boyas seleccionadas.
- `dataset_complementario_exterior.csv`: incluye variables de oleaje, viento y otras variables océano-meteorológicas de las boyas exteriores.

El análisis no pretende comparar Gijón y Bilbao de forma competitiva, sino mostrar cómo los datos objetivos de clima marítimo pueden aportar información útil para la gestión portuaria.

In [2]:
# ============================================
# 01. Carga, limpieza y normalización de datos
# TFM - Big Data aplicado al clima marítimo cantábrico
# ============================================

import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

Comprobación de archivos

In [4]:
pd.read_csv("/content/REDEXT2242_WAVEGIJON.csv", sep="\t", skiprows=1)

,Fecha (GMT),Altura Signif. del Oleaje(m),Periodo Medio Tm02(s),Periodo de Pico(s),Altura Máxima del Oleaje(m),Periodo de la Ola Maxima(s),Canal de obtencion de los datos,"Direcc. Media de Proced.(0=N,90=E)","Direcc. de pico de proced.(0=N,90=E)",Dispersión angular en el pico de energía espectral(grados),Canal de obtencion de los datos
0,2004 12 31 00,2.57,9.15,12.50,3.87,11.01,1,321.0,327.0,18.0,1
1,2004 12 31 01,2.22,8.69,11.11,3.60,11.68,1,321.0,321.0,21.0,1
2,2004 12 31 02,2.45,9.02,11.72,3.56,12.16,1,322.0,320.0,14.0,1
3,2004 12 31 03,2.21,8.43,11.18,3.32,9.53,1,320.0,316.0,22.0,1
4,2004 12 31 04,2.22,8.55,10.49,2.85,12.50,1,320.0,316.0,17.0,1
...,...,...,...,...,...,...,...,...,...,...,...
155930,2024 12 31 19,1.44,8.05,11.11,2.15,11.00,1,320.0,316.0,24.0,1
155931,2024 12 31 20,1.32,7.50,11.10,2.30,10.76,1,323.0,317.0,29.0,1
155932,2024 12 31 21,1.25,7.51,10.75,1.91,11.17,1,319.0,315.0,33.0,1
155933,2024 12 31 22,1.36,7.69,10.46,1.95,11.35,1,321.0,313.0,27.0,1


Comprobación total de archivos cargados ( 6 datasets en total)

In [5]:
import os

print("Archivos CSV disponibles en /content/:")
for archivo in os.listdir("/content"):
    if archivo.endswith(".csv"):
        print(archivo)

Archivos CSV disponibles en /content/:
REDEXT2242_WAVEGIJON.csv
REDEXT2136_WAVEBILBAO.csv
REDCOST1103_WAVBILBAO.csv
REDCOST1117_WAVEGIJON.csv
REDEXT2136_ALLBILBAO.csv
REDEXT2242_ALLGIJON.csv


Importación de librerías y creación de carpetas de salida

In [6]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Carpetas dentro de la sesión de Colab
RAW_DIR = Path("/content")
PROCESSED_DIR = Path("/content/processed")
TABLES_DIR = Path("/content/tables")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos originales:", RAW_DIR)
print("Carpeta de datos procesados:", PROCESSED_DIR)
print("Carpeta de tablas:", TABLES_DIR)

Carpeta de datos originales: /content
Carpeta de datos procesados: /content/processed
Carpeta de tablas: /content/tables


Definicitón de metadatos de los 6 data sets

In [8]:
datasets = {
    "REDCOST1117_WAVEGIJON.csv": {
        "codigo_boya": "1117",
        "nombre_boya": "Gijón costera",
        "zona": "Gijón",
        "tipo_red": "Red costera",
        "tipo_boya": "Costera",
        "tipo_dataset": "WAVE"
    },
    "REDEXT2242_WAVEGIJON.csv": {
        "codigo_boya": "2242",
        "nombre_boya": "Cabo de Peñas",
        "zona": "Asturias/Gijón",
        "tipo_red": "Red exterior",
        "tipo_boya": "Exterior",
        "tipo_dataset": "WAVE"
    },
    "REDCOST1103_WAVBILBAO.csv": {
        "codigo_boya": "1103",
        "nombre_boya": "Bilbao costera",
        "zona": "Bilbao",
        "tipo_red": "Red costera",
        "tipo_boya": "Costera",
        "tipo_dataset": "WAVE"
    },
    "REDEXT2136_WAVEBILBAO.csv": {
        "codigo_boya": "2136",
        "nombre_boya": "Bilbao-Vizcaya",
        "zona": "Bilbao-Vizcaya",
        "tipo_red": "Red exterior",
        "tipo_boya": "Exterior",
        "tipo_dataset": "WAVE"
    },
    "REDEXT2242_ALLGIJON.csv": {
        "codigo_boya": "2242",
        "nombre_boya": "Cabo de Peñas",
        "zona": "Asturias/Gijón",
        "tipo_red": "Red exterior",
        "tipo_boya": "Exterior",
        "tipo_dataset": "ALL"
    },
    "REDEXT2136_ALLBILBAO.csv": {
        "codigo_boya": "2136",
        "nombre_boya": "Bilbao-Vizcaya",
        "zona": "Bilbao-Vizcaya",
        "tipo_red": "Red exterior",
        "tipo_boya": "Exterior",
        "tipo_dataset": "ALL"
    }
}

Funciones de lectura y limpieza

In [10]:
def limpiar_nombre_columna(col):
    """
    Limpia nombres de columnas eliminando espacios raros, saltos y caracteres no deseados.
    """
    col = str(col).replace("\xa0", " ").strip()
    col = re.sub(r"\s+", " ", col)
    return col


def leer_csv_puertos(path):
    """
    Lee archivos CSV de Puertos del Estado.
    Los archivos vienen separados por tabulador y la primera fila indica el valor nulo.
    """
    df = pd.read_csv(
        path,
        sep="\t",
        skiprows=1,
        na_values=[-9999.9, "-9999.9"],
        engine="python"
    )

    df.columns = [limpiar_nombre_columna(c) for c in df.columns]
    return df


def convertir_fecha(df):
    """
    Convierte la columna Fecha (GMT) al formato datetime.
    """
    df["fecha"] = pd.to_datetime(
        df["Fecha (GMT)"],
        format="%Y %m %d %H",
        errors="coerce"
    )
    return df


def obtener_columna(df, posibles_nombres):
    """
    Devuelve la primera columna encontrada dentro de una lista de posibles nombres.
    """
    for nombre in posibles_nombres:
        if nombre in df.columns:
            return nombre
    return None


def convertir_numerica(df, columna):
    """
    Convierte una columna a numérica si existe.
    Si la columna no existe, devuelve una serie de NaN.
    """
    if columna is None:
        return pd.Series([np.nan] * len(df))
    return pd.to_numeric(df[columna], errors="coerce")

Cargar los 6 datasets

In [11]:
raw_data = {}
resumen_archivos = []

for file_name, meta in datasets.items():
    path = RAW_DIR / file_name

    print("\nLeyendo:", file_name)

    if not path.exists():
        print("No se encuentra el archivo:", path)
        continue

    df = leer_csv_puertos(path)
    df = convertir_fecha(df)

    raw_data[file_name] = df

    resumen_archivos.append({
        "archivo": file_name,
        "codigo_boya": meta["codigo_boya"],
        "nombre_boya": meta["nombre_boya"],
        "zona": meta["zona"],
        "tipo_red": meta["tipo_red"],
        "tipo_boya": meta["tipo_boya"],
        "tipo_dataset": meta["tipo_dataset"],
        "registros": len(df),
        "columnas": df.shape[1],
        "fecha_inicio": df["fecha"].min(),
        "fecha_fin": df["fecha"].max(),
        "fechas_nulas": df["fecha"].isna().sum()
    })

resumen_archivos_df = pd.DataFrame(resumen_archivos)
resumen_archivos_df


Leyendo: REDCOST1117_WAVEGIJON.csv

Leyendo: REDEXT2242_WAVEGIJON.csv

Leyendo: REDCOST1103_WAVBILBAO.csv

Leyendo: REDEXT2136_WAVEBILBAO.csv

Leyendo: REDEXT2242_ALLGIJON.csv

Leyendo: REDEXT2136_ALLBILBAO.csv


,archivo,codigo_boya,nombre_boya,zona,tipo_red,tipo_boya,tipo_dataset,registros,columnas,fecha_inicio,fecha_fin,fechas_nulas
0,REDCOST1117_WAVEGIJON.csv,1117,Gijón costera,Gijón,Red costera,Costera,WAVE,150719,15,2005-01-05 09:00:00,2024-12-31 23:00:00,0
1,REDEXT2242_WAVEGIJON.csv,2242,Cabo de Peñas,Asturias/Gijón,Red exterior,Exterior,WAVE,155935,12,2004-12-31 00:00:00,2024-12-31 23:00:00,0
2,REDCOST1103_WAVBILBAO.csv,1103,Bilbao costera,Bilbao,Red costera,Costera,WAVE,140991,15,2005-01-01 00:00:00,2024-12-31 23:00:00,0
3,REDEXT2136_WAVEBILBAO.csv,2136,Bilbao-Vizcaya,Bilbao-Vizcaya,Red exterior,Exterior,WAVE,162793,12,2005-01-01 00:00:00,2024-12-31 23:00:00,0
4,REDEXT2242_ALLGIJON.csv,2242,Cabo de Peñas,Asturias/Gijón,Red exterior,Exterior,ALL,156065,22,2005-01-01 00:00:00,2024-12-31 23:00:00,0
5,REDEXT2136_ALLBILBAO.csv,2136,Bilbao-Vizcaya,Bilbao-Vizcaya,Red exterior,Exterior,ALL,164084,22,2005-01-01 00:00:00,2024-12-30 23:00:00,0


In [13]:
resumen_archivos_df.to_csv(TABLES_DIR / "tabla_resumen_archivos.csv", index=False)

Revisión de columnas originales

In [14]:
for file_name, df in raw_data.items():
    print("\n" + "="*90)
    print(file_name)
    print("="*90)
    for i, col in enumerate(df.columns):
        print(i, col)


REDCOST1117_WAVEGIJON.csv
0 Fecha (GMT)
1 Altura Signif. del Oleaje (Hm0)(m)
2 Altura signif. de cruce por cero (H1/3)(m)
3 Periodo Medio(s)
4 Periodo Medio Tm02(s)
5 Periodo de Pico(s)
6 Altura Máxima del Oleaje(m)
7 Periodo de la Ola Maxima(s)
8 Canal de obtencion de los datos
9 Direcc. Media de Proced.(0=N,90=E)
10 Direccion Media en el Pico Espectral(0=N,90=E)
11 Dispersión angular en toda la banda resuelta(grados)
12 Dispersión angular en el pico de energía espectral(grados)
13 Canal de obtencion de los datos.1
14 fecha

REDEXT2242_WAVEGIJON.csv
0 Fecha (GMT)
1 Altura Signif. del Oleaje(m)
2 Periodo Medio Tm02(s)
3 Periodo de Pico(s)
4 Altura Máxima del Oleaje(m)
5 Periodo de la Ola Maxima(s)
6 Canal de obtencion de los datos
7 Direcc. Media de Proced.(0=N,90=E)
8 Direcc. de pico de proced.(0=N,90=E)
9 Dispersión angular en el pico de energía espectral(grados)
10 Canal de obtencion de los datos
11 fecha

REDCOST1103_WAVBILBAO.csv
0 Fecha (GMT)
1 Altura Signif. del Oleaje (Hm0)(m)

Función para normalizar el oleaje

In [16]:
def construir_dataset_oleaje(df, meta):
    """
    Construye un dataset normalizado de oleaje a partir de un dataset original.
    """

    col_hs = obtener_columna(df, [
        "Altura Signif. del Oleaje(m)",
        "Altura Signif. del Oleaje (Hm0)(m)"
    ])

    col_h1_3 = obtener_columna(df, [
        "Altura signif. de cruce por cero (H1/3)(m)"
    ])

    col_hmax = obtener_columna(df, [
        "Altura Máxima del Oleaje(m)"
    ])

    col_tm02 = obtener_columna(df, [
        "Periodo Medio Tm02(s)"
    ])

    col_tp = obtener_columna(df, [
        "Periodo de Pico(s)"
    ])

    col_dir_media = obtener_columna(df, [
        "Direcc. Media de Proced.(0=N,90=E)"
    ])

    col_dir_pico = obtener_columna(df, [
        "Direcc. de pico de proced.(0=N,90=E)",
        "Direccion Media en el Pico Espectral(0=N,90=E)"
    ])

    col_disp_pico = obtener_columna(df, [
        "Dispersión angular en el pico de energía espectral(grados)"
    ])

    out = pd.DataFrame({
        "fecha": df["fecha"],
        "codigo_boya": meta["codigo_boya"],
        "nombre_boya": meta["nombre_boya"],
        "zona": meta["zona"],
        "tipo_red": meta["tipo_red"],
        "tipo_boya": meta["tipo_boya"],
        "Hs_m": convertir_numerica(df, col_hs),
        "H1_3_m": convertir_numerica(df, col_h1_3),
        "Hmax_m": convertir_numerica(df, col_hmax),
        "Tm02_s": convertir_numerica(df, col_tm02),
        "Tp_s": convertir_numerica(df, col_tp),
        "dir_media_oleaje_grados": convertir_numerica(df, col_dir_media),
        "dir_pico_oleaje_grados": convertir_numerica(df, col_dir_pico),
        "dispersion_pico_grados": convertir_numerica(df, col_disp_pico)
    })

    return out

In [ ]:
Crear data set principal de oleaje con la red exterior y la red costera

In [17]:
wave_dfs = []

for file_name, meta in datasets.items():
    if meta["tipo_dataset"] == "WAVE":
        print("Normalizando oleaje:", file_name)
        df = raw_data[file_name]
        wave_dfs.append(construir_dataset_oleaje(df, meta))

dataset_oleaje = pd.concat(wave_dfs, ignore_index=True)
dataset_oleaje = dataset_oleaje.dropna(subset=["fecha"])
dataset_oleaje = dataset_oleaje.sort_values(["codigo_boya", "fecha"]).reset_index(drop=True)

display(dataset_oleaje.head())
print(dataset_oleaje.shape)

Normalizando oleaje: REDCOST1117_WAVEGIJON.csv
Normalizando oleaje: REDEXT2242_WAVEGIJON.csv
Normalizando oleaje: REDCOST1103_WAVBILBAO.csv
Normalizando oleaje: REDEXT2136_WAVEBILBAO.csv


,fecha,codigo_boya,nombre_boya,zona,tipo_red,tipo_boya,Hs_m,H1_3_m,Hmax_m,Tm02_s,Tp_s,dir_media_oleaje_grados,dir_pico_oleaje_grados,dispersion_pico_grados
0,2005-01-01 00:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,1.70,1.61,2.36,7.99,11.71,329.0,326.0,31.0
1,2005-01-01 01:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,2.03,1.89,2.94,8.44,11.70,328.0,326.0,40.0
2,2005-01-01 02:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,1.87,1.77,2.54,8.39,11.76,327.0,326.0,36.0
3,2005-01-01 03:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,2.04,1.92,2.72,8.71,12.48,329.0,333.0,36.0
4,2005-01-01 04:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,1.92,1.82,2.59,8.60,11.79,324.0,325.0,45.0


(610438, 14)


Se guarda el data set, Este dataset tiene las cuatro boyas de oleaje:

*  1117 - Gijón costera
*  2242 - Cabo de Peñas exterior
*  1103 - Bilbao costera
*  2136 - Bilbao-Vizcaya exterior

Se utilizará para un uso de datos objetivos para caracterizar distintas condiciones marítimas del Cantábrico y mostrar cómo esta información puede apoyar la gestión portuaria.



In [19]:
dataset_oleaje.to_csv(PROCESSED_DIR / "dataset_principal_oleaje.csv", index=False)

Se compureba el encabezado del data set generado

In [20]:
dataset_oleaje.head()

,fecha,codigo_boya,nombre_boya,zona,tipo_red,tipo_boya,Hs_m,H1_3_m,Hmax_m,Tm02_s,Tp_s,dir_media_oleaje_grados,dir_pico_oleaje_grados,dispersion_pico_grados
0,2005-01-01 00:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,1.70,1.61,2.36,7.99,11.71,329.0,326.0,31.0
1,2005-01-01 01:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,2.03,1.89,2.94,8.44,11.70,328.0,326.0,40.0
2,2005-01-01 02:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,1.87,1.77,2.54,8.39,11.76,327.0,326.0,36.0
3,2005-01-01 03:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,2.04,1.92,2.72,8.71,12.48,329.0,333.0,36.0
4,2005-01-01 04:00:00,1103,Bilbao costera,Bilbao,Red costera,Costera,1.92,1.82,2.59,8.60,11.79,324.0,325.0,45.0


Se contruye el data set complementario de datos de viento boyas exteriores:
* 2242, Cabo de Peñas
* 2136, Bilbao-Vizcaya

Este dataset incluirá oleaje, viento, presión, temperatura, salinidad y corrientes.

In [21]:
def construir_dataset_exterior(df, meta):
    """
    Construye un dataset normalizado con oleaje, viento y variables océano-meteorológicas
    disponibles en las boyas exteriores.
    """

    # Primero reutilizamos la estructura normalizada de oleaje
    base = construir_dataset_oleaje(df, meta)

    col_temp_agua = obtener_columna(df, [
        "Temperatura del Agua(ºC)"
    ])

    col_salinidad = obtener_columna(df, [
        "Salinidad(psu)"
    ])

    col_vel_corriente = obtener_columna(df, [
        "Velocidad media de Corriente(cm/s)"
    ])

    col_dir_corriente = obtener_columna(df, [
        "Dir. de prop. de la Corriente(0=N,90=E)"
    ])

    col_presion = obtener_columna(df, [
        "Presion atmosférica(hpa)"
    ])

    col_temp_aire = obtener_columna(df, [
        "Temperatura del Aire(ºC)"
    ])

    col_vel_viento = obtener_columna(df, [
        "Velocidad media del viento(m/s)"
    ])

    col_dir_viento = obtener_columna(df, [
        "Direc. de proced. del Viento(0=N,90=E)"
    ])

    base["temperatura_agua_c"] = convertir_numerica(df, col_temp_agua)
    base["salinidad_psu"] = convertir_numerica(df, col_salinidad)
    base["velocidad_corriente_cms"] = convertir_numerica(df, col_vel_corriente)
    base["dir_corriente_grados"] = convertir_numerica(df, col_dir_corriente)
    base["presion_hpa"] = convertir_numerica(df, col_presion)
    base["temperatura_aire_c"] = convertir_numerica(df, col_temp_aire)
    base["velocidad_viento_ms"] = convertir_numerica(df, col_vel_viento)
    base["dir_viento_grados"] = convertir_numerica(df, col_dir_viento)

    return base

Creación del dataset para tratamiento de datos complementarios (viento)

In [22]:
ext_dfs = []

for file_name, meta in datasets.items():
    if meta["tipo_dataset"] == "ALL":
        print("Normalizando dataset exterior:", file_name)
        df = raw_data[file_name]
        ext_dfs.append(construir_dataset_exterior(df, meta))

dataset_exterior = pd.concat(ext_dfs, ignore_index=True)
dataset_exterior = dataset_exterior.dropna(subset=["fecha"])
dataset_exterior = dataset_exterior.sort_values(["codigo_boya", "fecha"]).reset_index(drop=True)

display(dataset_exterior.head())
print(dataset_exterior.shape)

Normalizando dataset exterior: REDEXT2242_ALLGIJON.csv
Normalizando dataset exterior: REDEXT2136_ALLBILBAO.csv


,fecha,codigo_boya,nombre_boya,zona,tipo_red,tipo_boya,Hs_m,H1_3_m,Hmax_m,Tm02_s,Tp_s,dir_media_oleaje_grados,dir_pico_oleaje_grados,dispersion_pico_grados,temperatura_agua_c,salinidad_psu,velocidad_corriente_cms,dir_corriente_grados,presion_hpa,temperatura_aire_c,velocidad_viento_ms,dir_viento_grados
0,2005-01-01 00:00:00,2136,Bilbao-Vizcaya,Bilbao-Vizcaya,Red exterior,Exterior,3.07,NaN,4.71,10.42,12.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1034.0,12.0,NaN,NaN
1,2005-01-01 01:00:00,2136,Bilbao-Vizcaya,Bilbao-Vizcaya,Red exterior,Exterior,2.84,NaN,4.25,10.05,14.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1034.0,12.1,NaN,NaN
2,2005-01-01 02:00:00,2136,Bilbao-Vizcaya,Bilbao-Vizcaya,Red exterior,Exterior,2.67,NaN,4.72,9.79,11.63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1034.0,12.2,0.7,NaN
3,2005-01-01 03:00:00,2136,Bilbao-Vizcaya,Bilbao-Vizcaya,Red exterior,Exterior,2.77,NaN,3.56,9.81,11.63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1033.0,12.0,2.3,NaN
4,2005-01-01 04:00:00,2136,Bilbao-Vizcaya,Bilbao-Vizcaya,Red exterior,Exterior,2.81,NaN,4.25,9.77,12.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1033.0,11.9,1.6,NaN


(320149, 22)


Se guarda el dataset generado

In [23]:
dataset_exterior.to_csv(PROCESSED_DIR / "dataset_complementario_exterior.csv", index=False)

print("Dataset exterior guardado en:")
print(PROCESSED_DIR / "dataset_complementario_exterior.csv")

Dataset exterior guardado en:
/content/processed/dataset_complementario_exterior.csv


Resumen: Se han creado dos data sets. El data set dataset_principal_oleaje.csv es la base principal del presente TFM, ya que permite analizar las cuatro boyas de oleaje de forma homogénea. El segundo se utilizará únicamente para análisis complementarios en boyas exteriores, especialmente viento y condiciones combinadas de mar abierto.

# Generación de tablas a incluir en la memoria del TFM.

In [26]:
def resumen_cobertura(df, variables):
    """
    Genera una tabla resumen con cobertura temporal y porcentaje de datos válidos
    por boya y variable.
    """

    filas = []

    for codigo, grupo in df.groupby("codigo_boya"):
        fila = {
            "codigo_boya": codigo,
            "nombre_boya": grupo["nombre_boya"].iloc[0],
            "zona": grupo["zona"].iloc[0],
            "tipo_red": grupo["tipo_red"].iloc[0],
            "tipo_boya": grupo["tipo_boya"].iloc[0],
            "fecha_inicio": grupo["fecha"].min(),
            "fecha_fin": grupo["fecha"].max(),
            "registros": len(grupo)
        }

        for var in variables:
            if var in grupo.columns:
                fila[f"{var}_validos"] = grupo[var].notna().sum()
                fila[f"{var}_pct_validos"] = round(grupo[var].notna().mean() * 100, 2)

        filas.append(fila)

    return pd.DataFrame(filas)

Cobertura data set principal de oleaje

In [27]:
variables_oleaje = [
    "Hs_m",
    "Hmax_m",
    "Tm02_s",
    "Tp_s",
    "dir_media_oleaje_grados",
    "dir_pico_oleaje_grados",
    "dispersion_pico_grados"
]

cobertura_oleaje = resumen_cobertura(dataset_oleaje, variables_oleaje)

display(cobertura_oleaje)

cobertura_oleaje.to_csv(TABLES_DIR / "tabla_cobertura_oleaje.csv", index=False)

print("Tabla de cobertura de oleaje guardada en:")
print(TABLES_DIR / "tabla_cobertura_oleaje.csv")

,codigo_boya,nombre_boya,zona,tipo_red,tipo_boya,fecha_inicio,fecha_fin,registros,Hs_m_validos,Hs_m_pct_validos,Hmax_m_validos,Hmax_m_pct_validos,Tm02_s_validos,Tm02_s_pct_validos,Tp_s_validos,Tp_s_pct_validos,dir_media_oleaje_grados_validos,dir_media_oleaje_grados_pct_validos,dir_pico_oleaje_grados_validos,dir_pico_oleaje_grados_pct_validos,dispersion_pico_grados_validos,dispersion_pico_grados_pct_validos
0,1103,Bilbao costera,Bilbao,Red costera,Costera,2005-01-01 00:00:00,2024-12-31 23:00:00,140991,140988,100.0,140991,100.00,140931,99.96,140988,100.0,134118,95.13,133818,94.91,133818,94.91
1,1117,Gijón costera,Gijón,Red costera,Costera,2005-01-05 09:00:00,2024-12-31 23:00:00,150719,150719,100.0,150719,100.00,150714,100.00,150719,100.0,150714,100.00,150369,99.77,150369,99.77
2,2136,Bilbao-Vizcaya,Bilbao-Vizcaya,Red exterior,Exterior,2005-01-01 00:00:00,2024-12-31 23:00:00,162793,162789,100.0,156856,96.35,162785,100.00,162789,100.0,160600,98.65,160600,98.65,152682,93.79
3,2242,Cabo de Peñas,Asturias/Gijón,Red exterior,Exterior,2004-12-31 00:00:00,2024-12-31 23:00:00,155935,155935,100.0,154710,99.21,155933,100.00,155935,100.0,155932,100.00,155932,100.00,154708,99.21


Tabla de cobertura de oleaje guardada en:
/content/tables/tabla_cobertura_oleaje.csv


Cobertura dataset complementario exterior

In [28]:
variables_exterior = [
    "Hs_m",
    "Hmax_m",
    "Tm02_s",
    "Tp_s",
    "dir_media_oleaje_grados",
    "dir_pico_oleaje_grados",
    "velocidad_viento_ms",
    "dir_viento_grados",
    "presion_hpa",
    "temperatura_aire_c",
    "temperatura_agua_c",
    "salinidad_psu",
    "velocidad_corriente_cms",
    "dir_corriente_grados"
]

cobertura_exterior = resumen_cobertura(dataset_exterior, variables_exterior)

display(cobertura_exterior)

cobertura_exterior.to_csv(TABLES_DIR / "tabla_cobertura_exterior.csv", index=False)

print("Tabla de cobertura exterior guardada en:")
print(TABLES_DIR / "tabla_cobertura_exterior.csv")

,codigo_boya,nombre_boya,zona,tipo_red,tipo_boya,fecha_inicio,fecha_fin,registros,Hs_m_validos,Hs_m_pct_validos,Hmax_m_validos,Hmax_m_pct_validos,Tm02_s_validos,Tm02_s_pct_validos,Tp_s_validos,Tp_s_pct_validos,dir_media_oleaje_grados_validos,dir_media_oleaje_grados_pct_validos,dir_pico_oleaje_grados_validos,dir_pico_oleaje_grados_pct_validos,velocidad_viento_ms_validos,velocidad_viento_ms_pct_validos,dir_viento_grados_validos,dir_viento_grados_pct_validos,presion_hpa_validos,presion_hpa_pct_validos,temperatura_aire_c_validos,temperatura_aire_c_pct_validos,temperatura_agua_c_validos,temperatura_agua_c_pct_validos,salinidad_psu_validos,salinidad_psu_pct_validos,velocidad_corriente_cms_validos,velocidad_corriente_cms_pct_validos,dir_corriente_grados_validos,dir_corriente_grados_pct_validos
0,2136,Bilbao-Vizcaya,Bilbao-Vizcaya,Red exterior,Exterior,2005-01-01,2024-12-30 23:00:00,164084,162765,99.2,156832,95.58,162761,99.19,162765,99.2,160576,97.86,160576,97.86,159827,97.41,151969,92.62,158714,96.73,156048,95.10,140156,85.42,128629,78.39,135455,82.55,136565,83.23
1,2242,Cabo de Peñas,Asturias/Gijón,Red exterior,Exterior,2005-01-01,2024-12-31 23:00:00,156065,155911,99.9,154686,99.12,155909,99.90,155911,99.9,155908,99.90,155908,99.90,149665,95.90,145919,93.50,149158,95.57,146172,93.66,149502,95.79,133126,85.30,147848,94.73,151288,96.94


Tabla de cobertura exterior guardada en:
/content/tables/tabla_cobertura_exterior.csv


Comprobación de archivos generados

In [29]:
print("Archivos generados en /content/processed:")
for archivo in os.listdir(PROCESSED_DIR):
    print("-", archivo)

print("\nArchivos generados en /content/tables:")
for archivo in os.listdir(TABLES_DIR):
    print("-", archivo)

Archivos generados en /content/processed:
- dataset_principal_oleaje.csv
- dataset_complementario_exterior.csv

Archivos generados en /content/tables:
- tabla_cobertura_oleaje.csv
- tabla_resumen_archivos.csv
- tabla_cobertura_exterior.csv
